### RECOMPTE DE PUNTS CAP DE SETMANA F1

In [39]:
#%pip install fastf1

In [40]:
# Imports
import fastf1 as f1
import pandas as pd

In [41]:
# Parameters
YEAR = 2025
GP = 18

#### FUNCTIONS

In [42]:
def clean_session_data(session_df):
    # Clean unnecessary columns from the session DataFrame
    session_df.drop(columns=['BroadcastName', 'DriverId', 'TeamColor', 'TeamId', 'FirstName', 'LastName', 'HeadshotUrl', 'CountryCode'], inplace=True)

    # TODO: Other cleaning??

    return session_df

def load_weekend_data(year, gp_number):
    gp = f1.get_event(year, gp_number)

    if gp.EventFormat == 'conventional':
        """ fp1 = gp.get_session('Practice 1')
        fp2 = gp.get_session('Practice 2')
        fp3 = gp.get_session('Practice 3') """
        ss = None
        sprint = None
        qualy = gp.get_session('Qualifying')
        race = gp.get_session('Race')
    else:
        """ fp1 = gp.get_session('Practice 1')
        fp2 = None
        fp3 = None """
        ss = gp.get_session('Sprint Shootout')
        sprint = gp.get_session('Sprint')
        qualy = gp.get_session('Qualifying')
        race = gp.get_session('Race')

    # Return loaded data as DataFrames
    """ if fp1 is not None:
        fp1.load()
        fp1_data = clean_session_data(fp1.results)
    else:
        fp1_data = None
    if fp2 is not None:
        fp2.load()
        fp2_data = clean_session_data(fp2.results)
    else:
        fp2_data = None
    if fp3 is not None:
        fp3.load()
        fp3_data = clean_session_data(fp3.results)
    else:
        fp3_data = None """
    if ss is not None:
        ss.load()
        ss_data = clean_session_data(ss.results)
    else:
        ss_data = None
    if sprint is not None:
        sprint.load()
        sprint_data = clean_session_data(sprint.results)
    else:
        sprint_data = None
    if qualy is not None:
        qualy.load()
        qualy_data = clean_session_data(qualy.results)
    else:
        qualy_data = None
    if race is not None:
        race.load()
        race_data = clean_session_data(race.results)
    else:
        race_data = None

    """ "FP1": fp1_data,
        "FP2": fp2_data,
        "FP3": fp3_data, """
    return {
        "SS": ss_data,
        "Sprint": sprint_data,
        "Qualifying": qualy_data,
        "Race": race_data
    }

In [43]:
# NORMATIVA: Recompte de punts per les porres
def calculate_race_points(user_top5, actual_results):
    """
    Calculate race points for a user.
    
    Parameters:
        user_top5 (list): List of 5 driver abbreviations in predicted order [P1, P2, P3, P4, P5]
        actual_results (DataFrame): Race results DataFrame from FastF1
    
    Returns:
        dict: Points breakdown and total
    """
    user_top3 = user_top5[:3]  # Extract top 3 from the top 5 prediction

    points = 0
    breakdown = {}

    # Get actual top 3 and top 5 (ordered)
    sorted_results = actual_results.sort_values('Position')
    actual_top3 = sorted_results['Abbreviation'].iloc[:3].tolist()
    actual_top5 = sorted_results['Abbreviation'].iloc[:5].tolist()

    # 5 points for exact top 3 (correct order)
    if user_top3 == actual_top3:
        points += 5
        breakdown['exact_top3'] = 5
    # 10 points for exact top 5 (correct order)
    elif user_top5 == actual_top5:
        points += 10
        breakdown['exact_top5'] = 10
    else:
        # 1 point for correct top 3 pilots (wrong order)
        top3_no_order = sum(1 for driver in user_top3 if driver in actual_top3)
        if top3_no_order == 3:
            points += 1
            breakdown['top3_no_order'] = 1

        # 1 point for correct top 5 pilots (wrong order)
        top5_no_order = sum(1 for driver in user_top5 if driver in actual_top5)
        if top5_no_order == 5:
            points += 1
            breakdown['top5_no_order'] = 1

    breakdown['total'] = points
    return breakdown


def calculate_quali_points(user_top3, actual_results):
    """
    Calculate qualifying points for a user.
    
    Parameters:
        user_top3 (list): List of 3 driver abbreviations in predicted order [P1, P2, P3]
        actual_results (DataFrame): Qualifying results DataFrame from FastF1
    
    Returns:
        dict: Points breakdown and total
    """
    points = 0
    breakdown = {}

    # Get actual top 3 (ordered)
    sorted_results = actual_results.sort_values('Position')
    actual_top3 = sorted_results['Abbreviation'].iloc[:3].tolist()

    # 3 points for exact top 3 (correct order)
    if user_top3 == actual_top3:
        points += 3
        breakdown['exact_top3'] = 3
    else:
        # 1 point for correct top 3 pilots (wrong order)
        top3_no_order = sum(1 for driver in user_top3 if driver in actual_top3)
        if top3_no_order == 3:
            points += 1
            breakdown['top3_no_order'] = 1

    breakdown['total'] = points
    return breakdown


def calculate_sprint_points(user_top5, actual_results):
    """
    Calculate sprint race points for a user.
    
    Parameters:
        user_top5 (list): List of 5 driver abbreviations in predicted order [P1, P2, P3, P4, P5]
        actual_results (DataFrame): Sprint results DataFrame from FastF1
    
    Returns:
        dict: Points breakdown and total
    """
    user_top3 = user_top5[:3]  # Extract top 3 from the top 5 prediction

    points = 0
    breakdown = {}

    # Get actual top 3 and top 5 (ordered)
    sorted_results = actual_results.sort_values('Position')
    actual_top3 = sorted_results['Abbreviation'].iloc[:3].tolist()
    actual_top5 = sorted_results['Abbreviation'].iloc[:5].tolist()

    # 3 points for exact top 3 (correct order)
    if user_top3 == actual_top3:
        points += 3
        breakdown['exact_top3'] = 3
    # 6 points for exact top 5 (correct order)
    elif user_top5 == actual_top5:
        points += 6
        breakdown['exact_top5'] = 6
    else:
        # 1 point for correct top 3 pilots (wrong order)
        top3_no_order = sum(1 for driver in user_top3 if driver in actual_top3)
        if top3_no_order == 3:
            points += 1
            breakdown['top3_no_order'] = 1

        # 1 point for correct top 5 pilots (wrong order)
        top5_no_order = sum(1 for driver in user_top5 if driver in actual_top5)
        if top5_no_order == 5:
            points += 1
            breakdown['top5_no_order'] = 1

    breakdown['total'] = points
    return breakdown


def calculate_sprint_shootout_points(user_top3, actual_results):
    """
    Calculate sprint shootout points for a user.
    Same logic as qualifying.
    """
    return calculate_quali_points(user_top3, actual_results)

In [44]:

def calculate_weekend_points(user_predictions, weekend_data):
    """
    Calculate total weekend points for a user.
    
    Parameters:
        user_predictions (dict): Dictionary with user predictions per session
            Example:
            {
                "Qualifying": {"top3": ["VER", "NOR", "LEC"]},
                "Race":        {"top5": ["VER", "NOR", "LEC", "SAI", "HAM"]},
                "Sprint":      {"top5": ["VER", "NOR", "LEC", "SAI", "HAM"]},
                "SS":          {"top3": ["VER", "NOR", "LEC"]}
            }
        weekend_data (dict): Dictionary with session DataFrames from load_weekend_data()
    
    Returns:
        dict: Points per session and total
    """
    total_points = 0
    results = {}

    if weekend_data["Qualifying"] is not None and "Qualifying" in user_predictions:
        q_points = calculate_quali_points(
            user_predictions["Qualifying"]["top3"],
            weekend_data["Qualifying"]
        )
        results["Qualifying"] = q_points
        total_points += q_points["total"]

    if weekend_data["Race"] is not None and "Race" in user_predictions:
        r_points = calculate_race_points(
            user_predictions["Race"]["top5"],
            weekend_data["Race"]
        )
        results["Race"] = r_points
        total_points += r_points["total"]

    if weekend_data["Sprint"] is not None and "Sprint" in user_predictions:
        s_points = calculate_sprint_points(
            user_predictions["Sprint"]["top5"],
            weekend_data["Sprint"]
        )
        results["Sprint"] = s_points
        total_points += s_points["total"]

    if weekend_data["SS"] is not None and "SS" in user_predictions:
        ss_points = calculate_sprint_shootout_points(
            user_predictions["SS"]["top3"],
            weekend_data["SS"]
        )
        results["SS"] = ss_points
        total_points += ss_points["total"]

    results["total"] = total_points
    return results

#### MAIN

In [45]:
# Load data for the specified weekend
weekend_data = load_weekend_data(YEAR, GP)

predictions = {
    "Pau": {
        "Qualifying": {"top3": ["RUS", "VER", "PIA"]},
        "Race":        {"top5": ["RUS", "VER", "NOR", "PIA", "ANT"]}
    },
    "David": {
        "Qualifying": {"top3": ["VER", "RUS", "PIA"]},
        "Race":        {"top5": ["RUS", "VER", "NOR", "ANT", "PIA"]}
    }
}

# Calculate points for each user
all_results = {}
for user, user_preds in predictions.items():
    all_results[user] = calculate_weekend_points(user_preds, weekend_data)

# Display results as DataFrame
results_df = pd.DataFrame({user: data["total"] for user, data in all_results.items()}, index=["Points"]).T
print(results_df.sort_values("Points", ascending=False))

core           INFO 	Loading data for Singapore Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['63', '1', '81', '12', '4', '44', '16', '6', '87', '14', '27', '30', '22', '5', '18', '43', '31', '10', '23', '55']
core           INFO 	Loading data for Singapore Grand Prix - Race [v3.8.1]
req            INFO 	Using cache

       Points
Pau         8
David       6
